# 23CSE301 Machine Learning — Review 1
# Regression Accumulated / Consolidated Results

This notebook is **only for the first 10 Regression algorithms**.

It reads the result CSV produced by each algorithm notebook and creates the consolidated outputs required for
the Review 1 regression rubric.

### Regression algorithms included

1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet Regression
5. Polynomial Regression
6. Decision Tree Regressor
7. Random Forest Regressor
8. Gradient Boosting Regressor
9. Support Vector Regressor (SVR)
10. K-Nearest Neighbors Regressor

**Do not run this notebook until the ten regression notebooks have been run and their result CSVs have been saved.**

## What this notebook produces

### C2 — Required 10-model comparison
A single table containing:
- Model
- R²
- RMSE
- MAE

All ten models are ranked by R².

### CV summary
The two best-performing regression models are shown with:
- Test R²
- Mean 5-fold CV R²
- CV standard deviation

### C3 — Hyperparameter tuning summary
Where a model notebook exported tuned results, this notebook shows:
- Baseline R²
- Tuned R²
- R² improvement
- Baseline/Tuned RMSE
- Baseline/Tuned MAE
- Best available parameter columns

This separation keeps the required C2 table clean instead of mixing tuning-only columns into the main ten-model table.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# Your Windows project folder:
WINDOWS_PROJECT = Path(r"C:\Sem-5\ml\jyp_notebook\Reg_review1")

# Fallbacks useful when the notebook is being tested in another environment.
candidates = [
    WINDOWS_PROJECT / "results",
    Path("../results"),
    Path("results"),
    Path("/mnt/data/results"),
]

RESULTS_DIR = next((p for p in candidates if p.exists() and p.is_dir()), None)

if RESULTS_DIR is None:
    raise FileNotFoundError(
        "Could not find the results folder. Expected it at "
        r"C:\Sem-5\ml\jyp_notebook\Reg_review1\results"
        " or a relative ../results folder."
    )

print("Using results folder:")
print(RESULTS_DIR.resolve())

Using results folder:
C:\Sem-5\ml\jyp_notebook\Reg_review1\results


## 1. Regression result-file audit

In [2]:
expected_files = {
    "regression_linear_results.csv": "Linear Regression",
    "regression_ridge_results.csv": "Ridge Regression",
    "regression_lasso_results.csv": "Lasso Regression",
    "regression_elasticnet_results.csv": "ElasticNet Regression",
    "regression_polynomial_results.csv": "Polynomial Regression",
    "regression_decision_tree_results.csv": "Decision Tree Regression",
    "regression_random_forest_results.csv": "Random Forest Regression",
    "regression_gradient_boosting_results.csv": "Gradient Boosting Regression",
    "regression_svr_results.csv": "Support Vector Regression",
    "regression_knn_results.csv": "KNN Regression",
}

audit_rows = []

for filename, model_name in expected_files.items():
    path = RESULTS_DIR / filename
    audit_rows.append({
        "Model": model_name,
        "Result file": filename,
        "Status": "FOUND" if path.exists() else "MISSING"
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

missing = audit_df.loc[audit_df["Status"] == "MISSING", "Model"].tolist()

if missing:
    print("\nMissing regression result files:")
    for model in missing:
        print(" -", model)
    print("\nRun the missing regression notebook(s), then rerun this accumulator.")
else:
    print("\nALL 10 REGRESSION RESULT FILES ARE PRESENT.")

,Model,Result file,Status
0,Linear Regression,regression_linear_results.csv,FOUND
1,Ridge Regression,regression_ridge_results.csv,FOUND
2,Lasso Regression,regression_lasso_results.csv,FOUND
3,ElasticNet Regression,regression_elasticnet_results.csv,FOUND
4,Polynomial Regression,regression_polynomial_results.csv,FOUND
5,Decision Tree Regression,regression_decision_tree_results.csv,FOUND
6,Random Forest Regression,regression_random_forest_results.csv,FOUND
7,Gradient Boosting Regression,regression_gradient_boosting_results.csv,FOUND
8,Support Vector Regression,regression_svr_results.csv,FOUND
9,KNN Regression,regression_knn_results.csv,FOUND



ALL 10 REGRESSION RESULT FILES ARE PRESENT.


## 2. Load and standardise the ten regression result files

In [3]:
records = []

for filename, expected_model in expected_files.items():
    path = RESULTS_DIR / filename

    if not path.exists():
        continue

    df_result = pd.read_csv(path)

    if df_result.empty:
        print(f"WARNING: {filename} is empty.")
        continue

    row = df_result.iloc[0].to_dict()

    # Always use the canonical model name for the consolidated table.
    row["Model"] = expected_model
    records.append(row)

if len(records) == 0:
    raise RuntimeError("No regression result CSVs were found.")

reg_all = pd.DataFrame(records)

print(f"Loaded {len(reg_all)}/10 regression result files.")
display(reg_all)

Loaded 10/10 regression result files.


,Model,R2,RMSE,MAE,Mean_CV_R2,Std_CV_R2,CV_R2_Mean,CV_R2_Std,Tuned_R2,Tuned_RMSE,Tuned_MAE,Best_k
0,Linear Regression,0.9357,1054.0894,678.6851,0.9360,0.0034,NaN,NaN,NaN,NaN,NaN,NaN
1,Ridge Regression,0.9351,1058.7873,676.8099,0.9360,0.0042,NaN,NaN,NaN,NaN,NaN,NaN
2,Lasso Regression,0.9357,1054.1204,678.6266,0.9360,0.0034,NaN,NaN,NaN,NaN,NaN,NaN
3,ElasticNet Regression,0.9323,1081.6298,678.2524,NaN,NaN,0.9340,0.0056,NaN,NaN,NaN,NaN
4,Polynomial Regression,0.9683,739.6388,461.9328,NaN,NaN,0.9578,0.0185,NaN,NaN,NaN,NaN
5,Decision Tree Regression,0.9300,1099.9062,632.4218,NaN,NaN,0.9122,0.0140,NaN,NaN,NaN,NaN
6,Random Forest Regression,0.9632,797.6028,430.2351,NaN,NaN,0.9559,0.0057,0.9618,812.3210,438.4860,NaN
7,Gradient Boosting Regression,0.9610,821.1617,454.6824,NaN,NaN,0.9593,0.0057,0.9663,762.9409,416.4928,NaN
8,Support Vector Regression,-0.0897,4338.3943,2670.4644,NaN,NaN,-0.0907,0.0146,0.9148,1213.0520,624.4021,NaN
9,KNN Regression,0.9096,1249.8040,679.4490,NaN,NaN,0.9150,0.0121,0.9136,1221.4521,680.5895,7.0000


## 3. C2 — Single consolidated 10-model comparison

This is the **main table to show for the Review 1 regression comparison**.
It intentionally contains only the rubric-required comparison metrics.

In [4]:
required = ["Model", "R2", "RMSE", "MAE"]

missing_required = [c for c in required if c not in reg_all.columns]

if missing_required:
    raise ValueError(
        f"The following required columns are missing from one or more result files: {missing_required}"
    )

comparison = (
    reg_all[required]
    .copy()
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

comparison.insert(0, "Rank", np.arange(1, len(comparison) + 1))

display(
    comparison.style.format({
        "R2": "{:.4f}",
        "RMSE": "{:.2f}",
        "MAE": "{:.2f}",
    })
)

if len(comparison) == 10:
    print("\nC2 CHECK: PASS — all 10 regression algorithms are present.")
else:
    print(f"\nC2 CHECK: INCOMPLETE — {len(comparison)}/10 regression algorithms are present.")

,Rank,Model,R2,RMSE,MAE
0,1,Polynomial Regression,0.9683,739.64,461.93
1,2,Random Forest Regression,0.9632,797.60,430.24
2,3,Gradient Boosting Regression,0.9610,821.16,454.68
3,4,Linear Regression,0.9357,1054.09,678.69
4,5,Lasso Regression,0.9357,1054.12,678.63
5,6,Ridge Regression,0.9351,1058.79,676.81
6,7,ElasticNet Regression,0.9323,1081.63,678.25
7,8,Decision Tree Regression,0.9300,1099.91,632.42
8,9,KNN Regression,0.9096,1249.80,679.45
9,10,Support Vector Regression,-0.0897,4338.39,2670.46



C2 CHECK: PASS — all 10 regression algorithms are present.


### C2 observation

The comparison shows noticeable differences in regression performance across the ten algorithms. Polynomial Regression gives the highest test R² of 0.9683, followed by Random Forest Regression at 0.9632 and Gradient Boosting Regression at 0.9610. The linear and regularised models, including Linear Regression, Ridge, Lasso and ElasticNet, achieve R² values between approximately 0.9323 and 0.9357. Decision Tree Regression and KNN Regression perform somewhat lower, while the default SVR configuration performs poorly with a negative R² of -0.0897. Overall, the results indicate that nonlinear and ensemble models capture the relationship between diamond characteristics and price better than the simpler linear baselines on this test split. The R² ranking and MAE ranking are not identical. Although Polynomial Regression has the highest R², Random Forest has the lowest MAE among the ten baseline models at approximately 430.24, whereas Polynomial Regression has an MAE of approximately 461.93. This shows why multiple evaluation metrics should be considered rather than relying on R² alone

## 4. Two best regression models — mandatory 5-fold CV R²

In [5]:
top_two = comparison.head(2)["Model"].tolist()

cv_columns = ["Model", "R2", "CV_R2_Mean", "CV_R2_Std"]

cv_available = [c for c in cv_columns if c in reg_all.columns]
top_two_cv = reg_all.loc[reg_all["Model"].isin(top_two), cv_available].copy()

# Preserve the same ranking order as the main comparison.
top_two_cv["RankOrder"] = top_two_cv["Model"].map(
    {name: i for i, name in enumerate(top_two)}
)
top_two_cv = (
    top_two_cv
    .sort_values("RankOrder")
    .drop(columns="RankOrder")
    .reset_index(drop=True)
)

display(
    top_two_cv.style.format({
        "R2": "{:.4f}",
        "CV_R2_Mean": "{:.4f}",
        "CV_R2_Std": "{:.4f}",
    })
)

,Model,R2,CV_R2_Mean,CV_R2_Std
0,Polynomial Regression,0.9683,0.9578,0.0185
1,Random Forest Regression,0.9632,0.9559,0.0057


### CV observation

Polynomial Regression has a held-out test R² of 0.9683 and a mean 5-fold CV R² of 0.9578, while Random Forest has a test R² of 0.9632 and a mean CV R² of 0.9559. The differences between test and mean CV performance are relatively small for both models, suggesting that their performance is reasonably consistent across different data splits. Random Forest has a lower CV standard deviation (0.0057) than Polynomial Regression (0.0185), indicating that its performance is more stable across the five folds.

## 5. C3 — Hyperparameter tuning summary

In [6]:
tuning_fields = [
    "Model",
    "R2", "Tuned_R2", "R2_Improvement",
    "RMSE", "Tuned_RMSE",
    "MAE", "Tuned_MAE",
    "Best_alpha",
    "Best_l1_ratio",
    "Best_max_depth",
    "Best_n_estimators",
    "Best_learning_rate",
    "Best_C",
    "Best_kernel",
    "Best_gamma",
    "Best_k",
    "Best_min_samples_split",
]

tuning = reg_all.copy()

# Calculate improvement whenever both baseline and tuned R² exist.
if "Tuned_R2" in tuning.columns:
    tuning["R2_Improvement"] = tuning["Tuned_R2"] - tuning["R2"]

available_tuning = [c for c in tuning_fields if c in tuning.columns]

tuning = tuning[available_tuning].copy()

if "Tuned_R2" in tuning.columns:
    tuning_rows = tuning[tuning["Tuned_R2"].notna()].copy()
else:
    tuning_rows = pd.DataFrame()

if tuning_rows.empty:
    print("No tuned result columns were found yet.")
else:
    display(tuning_rows.style.format({
        c: "{:.4f}" for c in [
            "R2", "Tuned_R2", "R2_Improvement"
        ] if c in tuning_rows.columns
    } | {
        c: "{:.2f}" for c in [
            "RMSE", "Tuned_RMSE", "MAE", "Tuned_MAE"
        ] if c in tuning_rows.columns
    }))

,Model,R2,Tuned_R2,R2_Improvement,RMSE,Tuned_RMSE,MAE,Tuned_MAE,Best_k
6,Random Forest Regression,0.9632,0.9618,-0.0014,797.60,812.32,430.24,438.49,nan
7,Gradient Boosting Regression,0.9610,0.9663,0.0053,821.16,762.94,454.68,416.49,nan
8,Support Vector Regression,-0.0897,0.9148,1.0045,4338.39,1213.05,2670.46,624.40,nan
9,KNN Regression,0.9096,0.9136,0.0041,1249.80,1221.45,679.45,680.59,7.000000


### C3 observation

| Model             | Baseline R² | Tuned R² | Improvement |
| ----------------- | ----------: | -------: | ----------: |
| Random Forest     |      0.9632 |   0.9618 |     -0.0014 |
| Gradient Boosting |      0.9610 |   0.9663 |     +0.0053 |
| SVR               |     -0.0897 |   0.9148 |     +1.0045 |
| KNN               |      0.9096 |   0.9136 |     +0.0040 |

Hyperparameter tuning produced different effects across the models. Gradient Boosting improved from an R² of 0.9610 to 0.9663, an improvement of approximately 0.0053, while KNN improved from 0.9096 to 0.9136. The largest numerical improvement was observed for SVR, where R² increased from -0.0897 to 0.9148, showing that the default configuration was poorly suited to the dataset and that the selected hyperparameters substantially improved its performance. Random Forest showed a small decrease from 0.9632 to 0.9618, meaning that the selected tuning configuration did not improve the held-out test result in this case.

## 6. Quick C3 requirement check

In [7]:
if "Tuned_R2" in reg_all.columns:
    tuned_models = reg_all[reg_all["Tuned_R2"].notna()].copy()
else:
    tuned_models = pd.DataFrame()

print(f"Tuned regression models detected: {len(tuned_models)}")

if len(tuned_models) >= 2:
    print("C3 CHECK: PASS — at least 2 regression models have tuned results.")
else:
    print("C3 CHECK: INCOMPLETE — fewer than 2 tuned regression models detected.")

Tuned regression models detected: 4
C3 CHECK: PASS — at least 2 regression models have tuned results.


## 7. Export the clean accumulated tables

In [8]:
# Save only when the relevant data exists.
comparison_path = RESULTS_DIR / "Review1_Regression_10_Model_Comparison.csv"
comparison.to_csv(comparison_path, index=False)

cv_path = RESULTS_DIR / "Review1_Regression_Top2_CV.csv"
top_two_cv.to_csv(cv_path, index=False)

if not tuning_rows.empty:
    tuning_path = RESULTS_DIR / "Review1_Regression_Tuning_Summary.csv"
    tuning_rows.to_csv(tuning_path, index=False)
    print("Saved:", tuning_path.resolve())

print("Saved:", comparison_path.resolve())
print("Saved:", cv_path.resolve())

Saved: C:\Sem-5\ml\jyp_notebook\Reg_review1\results\Review1_Regression_Tuning_Summary.csv
Saved: C:\Sem-5\ml\jyp_notebook\Reg_review1\results\Review1_Regression_10_Model_Comparison.csv
Saved: C:\Sem-5\ml\jyp_notebook\Reg_review1\results\Review1_Regression_Top2_CV.csv
